In [25]:
import numpy as np
from collections import Counter
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)
alloy = read('bulk_structure/Mg4Zn7_relaxed.cif')
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")
print(f"Cell: {np.round(alloy.cell.lengths(), 3)} A")
print(f"Angles: {np.round(alloy.cell.angles(), 2)} deg")

Bulk: 110 atoms, Mg40Zn70
Cell: [26.354  5.337 14.452] A
Angles: [ 90.  102.5  90. ] deg


In [26]:
slab = surface(alloy, (1, 0, 0), 2, vacuum=8)
sc = make_supercell(slab, [[2,0,0],[0,1,0],[0,0,1]])

sym = np.array(sc.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')

ps = AseAtomsAdaptor().get_structure(sc)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(1,0,0), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(sc)}, Mg: {mg}, Zn: {zn}")
print(f"Stoichiometric: {'YES' if zn*4 == mg*7 else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 440, Mg: 160, Zn: 280
Stoichiometric: YES
Symmetric: False


In [27]:
z = sc.get_positions()[:, 2]
sym = np.array(sc.get_chemical_symbols())
order = np.argsort(z)

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < 0.3:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

z_min, z_max = z.min(), z.max()
n = len(layers)

print(f"Total atomic layers: {n}\n")
print(f"{'Bot':>7} {'Bot Comp':>10} {'from_bot':>9}  |  {'Top':>7} {'Top Comp':>10} {'from_top':>9} {'Match':>6}")
print("-" * 80)
for i in range(min(10, n//2)):
    bot, top = layers[i], layers[n-1-i]
    zm_b = np.mean([z[j] for j in layer_idx[i]])
    zm_t = np.mean([z[j] for j in layer_idx[n-1-i]])
    
    mg_b, zn_b = bot.get('Mg',0), bot.get('Zn',0)
    mg_t, zn_t = top.get('Mg',0), top.get('Zn',0)
    comp_b = f"Mg{mg_b}Zn{zn_b}" if mg_b and zn_b else (f"Mg{mg_b}" if mg_b else f"Zn{zn_b}")
    comp_t = f"Mg{mg_t}Zn{zn_t}" if mg_t and zn_t else (f"Mg{mg_t}" if mg_t else f"Zn{zn_t}")
    
    match = "YES" if bot == top else "NO <---"
    print(f"{i:>7} {comp_b:>10} {zm_b-z_min:>9.3f}  |  {n-1-i:>7} {comp_t:>10} {z_max-zm_t:>9.3f} {match:>6}")

Total atomic layers: 57

    Bot   Bot Comp  from_bot  |      Top   Top Comp  from_top  Match
--------------------------------------------------------------------------------
      0     Mg2Zn2     0.050  |       56     Mg6Zn4     0.381 NO <---
      1        Mg2     0.503  |       55        Mg2     1.098    YES
      2        Mg2     1.067  |       54        Mg2     1.661    YES
      3       Zn16     1.873  |       53       Zn16     2.467    YES
      4        Mg2     2.740  |       52        Mg2     3.335    YES
      5     Mg8Zn8     3.956  |       51     Mg8Zn8     4.551    YES
      6        Mg2     5.115  |       50        Mg2     5.710    YES
      7       Zn16     6.043  |       49       Zn16     6.638    YES
      8        Mg2     6.971  |       48        Mg2     7.566    YES
      9     Mg8Zn8     8.130  |       47     Mg8Zn8     8.724    YES


In [28]:
print("Searching for removals that give Symmetric=True...\n")

for bot_rm in range(0, 4):
    for top_rm in range(0, 4):
        if bot_rm == 0 and top_rm == 0:
            continue
        
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) == 0:
            continue
        
        tr = sc[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(1,0,0), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            is_sym = slab_tr.is_symmetric()
        except:
            is_sym = False
        
        if is_sym:
            sym_tr = np.array(tr.get_chemical_symbols())
            print(f"  bot_rm={bot_rm}, top_rm={top_rm}: "
                  f"{len(tr)} atoms, Mg{sum(sym_tr=='Mg')} Zn{sum(sym_tr=='Zn')}, "
                  f"Symmetric=True")

Searching for removals that give Symmetric=True...

  bot_rm=1, top_rm=1: 426 atoms, Mg152 Zn274, Symmetric=True
  bot_rm=2, top_rm=2: 422 atoms, Mg148 Zn274, Symmetric=True
  bot_rm=3, top_rm=3: 418 atoms, Mg144 Zn274, Symmetric=True


In [29]:
removed_bot = layer_idx[0]
removed_top = layer_idx[n-1]
removed_all = removed_bot + removed_top

print(f"Bottom layer removed ({len(removed_bot)} atoms):")
for j in removed_bot:
    print(f"  idx={j} {sym[j]} ({sc.positions[j][0]:.3f}, {sc.positions[j][1]:.3f}, {sc.positions[j][2]:.3f})")

print(f"\nTop layer removed ({len(removed_top)} atoms):")
for j in removed_top:
    print(f"  idx={j} {sym[j]} ({sc.positions[j][0]:.3f}, {sc.positions[j][1]:.3f}, {sc.positions[j][2]:.3f})")

rem_mg = sum(sym[j] == 'Mg' for j in removed_all)
rem_zn = sum(sym[j] == 'Zn' for j in removed_all)
print(f"\nTotal removed: {len(removed_all)} atoms (Mg{rem_mg} Zn{rem_zn})")

# Get inversion from the symmetric sub-slab
keep = []
for j in range(1, n-1):
    keep.extend(layer_idx[j])
trimmed = sc[keep]

ps_trim = AseAtomsAdaptor().get_structure(trimmed)
sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"\nSymmetric sub-slab: SG = {sga.get_space_group_symbol()}")

for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

Bottom layer removed (4 atoms):
  idx=246 Mg (7.957, 5.604, 8.000)
  idx=26 Mg (2.620, 5.604, 8.000)
  idx=76 Zn (2.620, 11.733, 8.101)
  idx=296 Zn (7.957, 11.733, 8.101)

Top layer removed (10 atoms):
  idx=187 Zn (2.620, 5.684, 58.579)
  idx=407 Zn (7.957, 5.684, 58.579)
  idx=137 Mg (2.620, 11.813, 58.680)
  idx=357 Mg (7.957, 11.813, 58.680)
  idx=362 Mg (10.625, 4.375, 58.864)
  idx=142 Mg (5.288, 4.375, 58.864)
  idx=370 Zn (10.625, 10.230, 59.069)
  idx=150 Zn (5.288, 10.230, 59.069)
  idx=363 Mg (10.625, 1.634, 59.274)
  idx=143 Mg (5.288, 1.634, 59.274)

Total removed: 14 atoms (Mg8 Zn6)

Symmetric sub-slab: SG = P2/m
Inversion: f -> [0.         0.20518602 0.99116608] - f


In [30]:
ps_orig = AseAtomsAdaptor().get_structure(sc)

print("Inversion partners of removed atoms:\n")
for j in removed_all:
    frac = ps_orig[j].frac_coords
    cart = ps_orig.lattice.get_cartesian_coords(frac)
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    
    # Check if partner exists in trimmed slab
    keep_set = set(keep)
    dists = np.linalg.norm(sc.get_positions() - inv_cart, axis=1)
    min_d = min(dists[k] for k in keep_set)
    closest = min(keep_set, key=lambda k: dists[k])
    paired = f"paired idx={closest}" if min_d < 0.5 else "UNPAIRED"
    
    print(f"  idx={j} {sym[j]} ({cart[0]:.3f}, {cart[1]:.3f}, {cart[2]:.3f}) "
          f"-> inv: ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f}) {paired}")

Inversion partners of removed atoms:

  idx=246 Mg (7.957, 5.604, 8.000) -> inv: (2.717, 11.813, 58.680) UNPAIRED
  idx=26 Mg (2.620, 5.604, 8.000) -> inv: (8.053, 11.813, 58.680) UNPAIRED
  idx=76 Zn (2.620, 11.733, 8.101) -> inv: (8.053, 5.684, 58.579) UNPAIRED
  idx=296 Zn (7.957, 11.733, 8.101) -> inv: (2.717, 5.684, 58.579) UNPAIRED
  idx=187 Zn (2.620, 5.684, 58.579) -> inv: (8.053, 11.733, 8.101) UNPAIRED
  idx=407 Zn (7.957, 5.684, 58.579) -> inv: (2.717, 11.733, 8.101) UNPAIRED
  idx=137 Mg (2.620, 11.813, 58.680) -> inv: (8.053, 5.604, 8.000) UNPAIRED
  idx=357 Mg (7.957, 11.813, 58.680) -> inv: (2.717, 5.604, 8.000) UNPAIRED
  idx=362 Mg (10.625, 4.375, 58.864) -> inv: (0.048, 13.042, 7.816) UNPAIRED
  idx=142 Mg (5.288, 4.375, 58.864) -> inv: (5.385, 13.042, 7.816) UNPAIRED
  idx=370 Zn (10.625, 10.230, 59.069) -> inv: (0.048, 7.187, 7.611) UNPAIRED
  idx=150 Zn (5.288, 10.230, 59.069) -> inv: (5.385, 7.187, 7.611) UNPAIRED
  idx=363 Mg (10.625, 1.634, 59.274) -> inv: (0.04

In [31]:
mg_removed = [i for i in removed_all if sym[i] == 'Mg']
zn_removed = [i for i in removed_all if sym[i] == 'Zn']
print(f"Removed: {len(mg_removed)} Mg, {len(zn_removed)} Zn")
print(f"Searching all 4+4 Mg splits × 3+3 Zn splits × pairings...\n")

found = False
count = 0

for keep_mg in combinations(mg_removed, 4):
    move_mg = [i for i in mg_removed if i not in keep_mg]
    for keep_zn in combinations(zn_removed, 3):
        move_zn = [i for i in zn_removed if i not in keep_zn]
        
        for perm_mg in permutations(keep_mg):
            for perm_zn in permutations(keep_zn):
                sc_test = sc.copy()
                
                for m, k in zip(move_mg, perm_mg):
                    frac = ps_orig[k].frac_coords
                    inv_frac = (inv_translation - frac) % 1.0
                    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
                    sc_test.positions[m] = inv_cart
                
                for m, k in zip(move_zn, perm_zn):
                    frac = ps_orig[k].frac_coords
                    inv_frac = (inv_translation - frac) % 1.0
                    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
                    sc_test.positions[m] = inv_cart
                
                count += 1
                if count % 5000 == 0:
                    print(f"  tested {count}...")
                
                try:
                    ps_test = AseAtomsAdaptor().get_structure(sc_test)
                    slab_test = Slab(ps_test.lattice, ps_test.species, ps_test.frac_coords,
                        miller_index=(1,0,0), oriented_unit_cell=ps_test, shift=0,
                        scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
                    is_sym = slab_test.is_symmetric()
                except:
                    is_sym = False
                
                if is_sym:
                    print(f"\nFOUND after {count} tries!")
                    print(f"  Moved Mg: {list(zip(move_mg, perm_mg))}")
                    print(f"  Moved Zn: {list(zip(move_zn, perm_zn))}")
                    sc_fixed = sc_test
                    
                    sym_f = np.array(sc_fixed.get_chemical_symbols())
                    mg_f = sum(sym_f == 'Mg')
                    zn_f = sum(sym_f == 'Zn')
                    print(f"  Atoms: {len(sc_fixed)}, Mg: {mg_f}, Zn: {zn_f}")
                    print(f"  Stoichiometric: {'YES' if zn_f*4 == mg_f*7 else 'NO'}")
                    found = True
                    break
            if found:
                break
        if found:
            break
    if found:
        break

if not found:
    print(f"No solution found ({count} combinations tested)")

Removed: 8 Mg, 6 Zn
Searching all 4+4 Mg splits × 3+3 Zn splits × pairings...

  tested 5000...
  tested 10000...
  tested 15000...
  tested 20000...
  tested 25000...
  tested 30000...

FOUND after 30097 tries!
  Moved Mg: [(np.int64(137), np.int64(246)), (np.int64(357), np.int64(26)), (np.int64(142), np.int64(362)), (np.int64(143), np.int64(363))]
  Moved Zn: [(np.int64(296), np.int64(76)), (np.int64(187), np.int64(370)), (np.int64(407), np.int64(150))]
  Atoms: 440, Mg: 160, Zn: 280
  Stoichiometric: YES


In [32]:
ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
    miller_index=(1,0,0), oriented_unit_cell=ps_fixed, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

print(f"Atoms: {len(sc_fixed)}")
print(f"Mg: {sum(sym_f=='Mg')}, Zn: {sum(sym_f=='Zn')}")
print(f"Stoichiometric: {'YES' if sum(sym_f=='Zn')*4 == sum(sym_f=='Mg')*7 else 'NO'}")
print(f"Symmetric: {slab_fixed.is_symmetric()}")
print(f"Thickness: {thick:.1f} A")
print(f"Area: {area:.1f} A²")

view(sc_fixed)

Atoms: 440
Mg: 160, Zn: 280
Stoichiometric: YES
Symmetric: True
Thickness: 51.9 A
Area: 154.2 A²


<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [55]:
ps = AseAtomsAdaptor().get_structure(sc_fixed)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg4zn7_100_2layers_reconstructed_ase.data")

print(f"Saved: slabs/mg4zn7_100_2layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

Saved: slabs/mg4zn7_100_2layers_reconstructed_ase.data
  Atoms: 440
  Thickness: 51.9 A
  Area: 154.2 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [56]:
#unrelaxed surface energy calculation
E_slab = -550.54843     # eV
E_bulk_per_fu =  -141.72792 / 10  # eV per formula unit 
n = 440 / 11                 # formula units in slab = 32
A = 154.2             # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -14.172792 eV
n * E_bulk = -566.91168 eV
E_slab - n*E_bulk = 16.36325 eV
S = 0.053059 eV/Ang^2
S = 0.8501 J/m^2


In [1]:
#relaxed surface energy calculation
E_slab_relaxed =  -554.846129473564  # eV
E_bulk_per_fu = -141.72792 / 10  # eV per formula unit
n = 440 / 11                      # 32 formula units
A = 154.2               # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.8501
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 12.06555 eV
S relaxed = 0.039123 eV/Ang^2
S relaxed = 0.6268 J/m^2

Unrelaxed S = 0.8501 J/m^2
Relaxed S   = 0.6268 J/m^2
Relaxation energy = 0.2233 J/m^2
